# Дообучение и сравнение `/watch`-парсера

1. Загрузите **этот ноутбук** в Colab: `File → Upload notebook`.
2. Включите GPU: `Среда выполнения → Сменить тип среды → T4 GPU`.
3. Выполните ячейки по порядку. Первая ячейка установит зависимости и попросит
   перезапустить сеанс; после перезапуска ячейка с загрузкой попросит выбрать
   `watch_parser.zip`.

Архив `watch_parser.zip` лежит рядом с папкой проекта.

In [ ]:
!pip -q install -U "transformers>=4.56" "peft>=0.13" accelerate matplotlib
!pip -q uninstall -y torchao
# После этой ячейки: Среда выполнения → Перезапустить сеанс, затем выполнять со следующей.

## Загрузка проекта

После перезапуска выполните ячейку ниже. Она попросит выбрать
`watch_parser.zip` и распакует его в `/content`.

In [ ]:
import os
import subprocess

from google.colab import files

os.chdir('/content')
if not os.path.isdir('/content/finetuning/watch_parser'):
    print('Выберите файл watch_parser.zip')
    files.upload()
    subprocess.run(['unzip', '-q', 'watch_parser.zip', '-d', '/content'], check=True)

os.chdir('/content/finetuning/watch_parser')
print('проект:', os.getcwd())

In [ ]:
!python scripts/check_dataset.py
!python scripts/check_bot_compat.py

In [ ]:
!python training/train_lora.py \
  --train data/v1/train.jsonl \
  --valid data/v1/valid.jsonl \
  --out /content/adapter_watch_json \
  --run-dir /content/run_watch_json

In [ ]:
# 1. База без инструкций
!python scripts/eval_model.py --prompt-mode none --out results/baseline/base_none.json

# 2. База с проработанным промптом
!python scripts/eval_model.py --prompt-mode fewshot --out results/baseline/base_prompt.json

# 3. Дообученная модель с тем же промптом
!python scripts/eval_model.py \
  --adapter /content/adapter_watch_json \
  --prompt-mode fewshot \
  --out results/v1/tuned_prompt.json

In [ ]:
!python scripts/compare_results.py \
  --base-none results/baseline/base_none.json \
  --base-prompt results/baseline/base_prompt.json \
  --tuned results/v1/tuned_prompt.json \
  --out results/v1/compare.md

from IPython.display import Markdown, display

display(Markdown(open('results/v1/compare.md', encoding='utf-8').read()))

In [ ]:
!cd /content && zip -qr watch_results.zip \
  finetuning/watch_parser/results \
  run_watch_json \
  adapter_watch_json

from google.colab import files

files.download('/content/watch_results.zip')